# 00 · Explore Overture place categories

Lists the `basic_category` / `taxonomy` values present in the city extent and shows how the
service mapping (`config/scoring.yaml` + the city's `services:` overrides) classifies them.
Output: `data/interim/<slug>/service_review.csv`, the table used to set the include/exclude lists.

Overture removed the `categories` column in September 2026; this notebook uses `basic_category`
and `taxonomy.{primary, hierarchy, alternates}`. Matching is on `taxonomy.hierarchy` (primary +
ancestors) and `alternates`, because `basic_category` is too coarse (e.g. `food_and_beverage_store`
mixes grocery, liquor and tobacco).

In [ ]:
import os

import urban_access  # noqa: F401  (first: sets PROJ_DATA before geopandas/pyogrio load)
from urban_access.config import load_city

CITY = os.environ.get("CITY", "carson_city")
cfg = load_city(CITY)
cfg.ensure_dirs()
print(
    cfg.name, "| Overture", cfg.overture_release, "| ACS", cfg.acs_year, "| EPSG", cfg.analysis_epsg
)
print("raw:", cfg.raw_dir)

In [ ]:
import duckdb
import folium
import geopandas as gpd
import pandas as pd

from urban_access import census, extract
from urban_access.services import ASSIGNED, classify_places

pd.set_option("display.max_rows", 300, "display.width", 200, "display.max_colwidth", 60)

# extent = bbox ∪ scored block groups, buffered (same as the extract step)
census.run(cfg)
extent = census.scored_extent(cfg).buffered(cfg.edge_buffer_m, cfg.analysis_epsg)
places_path = extract.extract_all(cfg, extent, only=["places"])["places"]
places = gpd.read_parquet(places_path)
print(f"{len(places):,} places in extent {tuple(round(v, 4) for v in extent.as_tuple())}")

## Category values in the extent

In [ ]:
con = duckdb.connect()
con.register("places", places.drop(columns="geometry"))
print(
    con.execute("""
    SELECT operating_status, count(*) n FROM places GROUP BY 1 ORDER BY 2 DESC""").df()
)
print(places["confidence"].describe(percentiles=[0.1, 0.25, 0.5]).round(2))
basic = con.execute("""
    SELECT basic_category, count(*) n, round(avg(confidence), 2) mean_conf
    FROM places GROUP BY 1 ORDER BY n DESC""").df()
primary = con.execute("""
    SELECT taxonomy_primary, any_value(basic_category) basic_category, count(*) n
    FROM places GROUP BY 1 ORDER BY n DESC""").df()
basic.to_csv(cfg.interim_dir / "basic_category_counts.csv", index=False)
primary.to_csv(cfg.interim_dir / "taxonomy_primary_counts.csv", index=False)
print(f"{len(basic)} basic_category values, {len(primary)} taxonomy.primary values")
basic.head(40)

In [ ]:
# hierarchy tree (levels 1–3) for the branches that hold the five services
roots = [
    "health_care",
    "education",
    "shopping",
    "sports_and_recreation",
    "travel_and_transportation",
    "arts_and_entertainment",
    "geographic_entities",
]
lvl = con.execute("""
    SELECT taxonomy_hierarchy[1] l1, taxonomy_hierarchy[2] l2, taxonomy_hierarchy[3] l3, count(*) n
    FROM places GROUP BY ALL ORDER BY l1, l2, n DESC""").df()
print(lvl.groupby("l1")["n"].sum().sort_values(ascending=False).head(25))
lvl[
    lvl["l1"].isin(roots)
    & ~(
        lvl["l1"].eq("shopping")
        & ~lvl["l2"].isin(
            [
                "food_and_beverage_store",
                "market",
                "convenience_store",
                "discount_store",
                "department_store",
                "warehouse_club_store",
            ]
        )
    )
]

In [ ]:
# land_use classes that could supply park polygons (scoring.yaml services.parks.landuse)
lu = (
    extract.duckdb_connection()
    .execute(
        """SELECT subtype, class, count(*) n FROM read_parquet(?, hive_partitioning = 1)
       WHERE bbox.xmin <= ? AND bbox.xmax >= ? AND bbox.ymin <= ? AND bbox.ymax >= ?
         AND subtype IN ('park', 'recreation', 'protected', 'horticulture')
       GROUP BY ALL ORDER BY 1, 3 DESC""",
        [
            extract.overture_path(cfg.overture_release, "base", "land_use"),
            extent.lon_max,
            extent.lon_min,
            extent.lat_max,
            extent.lat_min,
        ],
    )
    .df()
)
print("configured:", cfg.scoring.services["parks"].landuse)
lu

## Classification review

`status`: `assigned` (counts as a service), `closed`, `low_confidence`, `excluded_name`,
`duplicate` (same normalised name within `dedupe_distance_m`). Edit the city's `services:`
include/exclude lists and re-run from here.

In [ ]:
cls = classify_places(places, cfg.scoring, cfg.analysis_epsg)
review = cls.assign(lon=cls.geometry.x.round(6), lat=cls.geometry.y.round(6)).drop(
    columns=["geometry", "name_norm"]
)
review.to_csv(cfg.interim_dir / "service_review.csv", index=False)
pd.crosstab(cls["service"], cls["status"], margins=True)

In [ ]:
cols = ["name", "taxonomy_primary", "rule", "status", "confidence", "operating_status"]
for svc in ["grocery", "schools", "healthcare"]:
    print(f"\n=== {svc}")
    print(cls.loc[cls["service"] == svc, cols].round(2).to_string())

In [ ]:
for svc in ["parks", "transit", "convenience"]:
    print(f"\n=== {svc}")
    print(cls.loc[cls["service"] == svc, cols].round(2).to_string())

Transit: Overture's transit POIs here are Greyhound stops and heritage-rail depots. Those
aren't usable for daily access, so transit comes from the JAC GTFS feed (notebook 02); Overture is
the fallback only for cities without `transit.gtfs_url`.

In [ ]:
final = cls[cls["status"] == ASSIGNED]
summary = final.groupby("service").size().rename("n_assigned").to_frame()
assert {"grocery", "healthcare", "schools", "parks"} <= set(summary.index), summary
summary

In [ ]:
colors = {
    "grocery": "green",
    "healthcare": "red",
    "schools": "blue",
    "parks": "darkgreen",
    "transit": "purple",
    "convenience": "orange",
}
m = folium.Map(location=cfg.bbox.centroid[::-1], zoom_start=12, tiles="CartoDB positron")
b = cfg.bbox
folium.Rectangle(
    [(b.lat_min, b.lon_min), (b.lat_max, b.lon_max)], color="black", fill=False, weight=1
).add_to(m)
for svc, grp in final.groupby("service"):
    layer = folium.FeatureGroup(name=f"{svc} ({len(grp)})")
    for _, r in grp.iterrows():
        folium.CircleMarker(
            (r.geometry.y, r.geometry.x),
            radius=4,
            color=colors.get(svc, "gray"),
            fill=True,
            tooltip=f"{svc}: {r['name']} ({r['taxonomy_primary']})",
        ).add_to(layer)
    layer.add_to(m)
folium.LayerControl().add_to(m)
m